# Converting coordinates from a csv file to a json file

In [124]:
import json
import os
import pandas as pd
import numpy as np
import rasterio

from PIL import Image
from pyproj import Transformer       

## Import the raster file to get the metadata

We need the information from the raster as the json file needs to be in reference to the origin and transformation of the tif file.

In [102]:
base_dir = 'images/labels/'
tif_name = '20230428_example'
tif_path = os.path.join(base_dir, tif_name + '.tif')

with rasterio.open(tif_path) as raster:
    # Read the raster band
    imported_raster = raster.read(1)
    # Get the metadata of the raster
    imported_raster_meta = raster.meta
    # Get the raster transform parameters
    raster_transform = raster.transform

print("Shape of the raster (rows, columns):")
print(imported_raster.shape)
print("\n")

print("Raster metadata:")
print(imported_raster_meta)
print("\n")

print("Affine transformation parameters:")
print(raster_transform)

Shape of the raster (rows, columns):
(6122, 9254)


Raster metadata:
{'driver': 'GTiff', 'dtype': 'uint16', 'nodata': 0.0, 'width': 9254, 'height': 6122, 'count': 4, 'crs': CRS.from_epsg(32753), 'transform': Affine(3.0, 0.0, 445854.0,
       0.0, -3.0, 8565888.0)}


Affine transformation parameters:
| 3.00, 0.00, 445854.00|
| 0.00,-3.00, 8565888.00|
| 0.00, 0.00, 1.00|


## Account for the png padding

As the png has padding, we need to figure out how much to add to the x and y coordinates to get the correct position for the labels in the png file.

This function will be different if there is padding on both sides and the top and bottom of the image. I've just accounted for padding on the left and top of the image by adding the difference in between the png size and the raster size to the x (difference in width) and y (difference in height) coordinates.

If the padding is consistent (e.g. tile_size - stride), we can just add that padding to the x and y coordinates. In that case, set width_diff and height_diff to the padding amount for the x and y axes respectively.

In [ ]:
# Load the png image
image = Image.open('images/labels/20230428_example.png')

# Convert the image to a numpy array
pixel_data = np.array(image)

# Get image information
png_width, png_height = image.size
print(f'Image size: ({png_width}, {png_height})')

# Calculate the different in the width and height of the image and the raster
width_diff = png_width - imported_raster.shape[1]
height_diff = png_height - imported_raster.shape[0]
print(f'Difference in width: {width_diff}')
print(f'Difference in height: {height_diff}')


# if using images with padded left and right (and top and bottom)
tile_size = 416
stride = 104
min_pad = tile_size - stride

width_diff = width_diff - min_pad
height_diff = height_diff - min_pad

Image size: (9984, 6656)
Difference in width: 730
Difference in height: 534


## Convert the locations in the csv file to the same projection as the tif file

As the locations are in WGS84, we need to convert them to the same projection as the tif file, and then convert them to pixel coordinates using the raster transformation from the metadata.

In [111]:
# Create the reprojection function
coord_transformer = Transformer.from_crs('epsg:4326', 'epsg:32753', always_xy=True)

# Test on sample set of coordinates
x,y = coord_transformer.transform(133.6153775, -13.7976017)
print(x,y)

# Check the pixel coordinates of the transformed coordinates
pixel_column, pixel_row = ~raster_transform * (x, y)
print(pixel_column, pixel_row)

350330.60260668746 8474226.385882989
-31841.132464437513 30553.871372337453


## Read in the csv file of labelled coordinates

In [139]:
# Specify the path to your CSV file
csv_base_dir = f'data/'
# csv_name = '2024 Rapid Waterhole Assessment_aligned.csv' # with health states
csv_name = 'Rapid Waterhole Assessment 2024.csv'
csv_file_path = os.path.join(csv_base_dir, csv_name)

# Read the CSV file into a DataFrame
waterhole_labelled_df = pd.read_csv(csv_file_path)

print(f'Number of samples: {len(waterhole_labelled_df)}')
print('\n')

# Display the first few rows of the DataFrame
print(waterhole_labelled_df.head())

Number of samples: 723


    Timestamp  Latitude  Longitude  Class        Observer  ObserverSi  \
0  2024-05-07 -13.65579   134.3538      2  Andrew Hoskins  Back Right   
1  2024-05-07 -13.65572   134.3702      2  Andrew Hoskins  Back Right   
2  2024-05-07 -13.65600   134.3936      3  Andrew Hoskins  Back Right   
3  2024-05-07 -13.65634   134.4219      2  Andrew Hoskins  Back Right   
4  2024-05-07 -13.65627   134.5256      2  Andrew Hoskins  Back Right   

  Water Type Chopper Water Type Satellite transectNu permanence    ID  \
0                NaN                River        m08          P         
1                NaN      River/Billabong        m08          P         
2                NaN            Billabong        m08     I or E         
3                NaN               Stream        m08     I or E  W269   
4                NaN            Billabong        m08     I or E         

  Wet Dry Chopper Wet Dry Satellite  
0             NaN               Wet  
1             NaN    

## Reproject the label coordinates to the same projection as the tif file

In [140]:
x_proj, y_proj = coord_transformer.transform(
    waterhole_labelled_df['Longitude'].values, 
    waterhole_labelled_df['Latitude'].values
)

print(x_proj, y_proj)

[430112.4198399  431886.15113605 434417.06104698 437477.92716284
 448693.42141363 457886.50185429 458254.20180212 458827.36309655
 461747.47334173 449407.23853638 445252.27894334 444159.50192696
 440794.70240962 435211.45049353 428860.19328362 425354.55770888
 424196.77749568 423212.16039536 421253.78933279 417110.28161243
 401196.41106128 391625.28131144 373867.42304861 371360.14238773
 358132.66349237 425752.81788484 426812.24890471 429614.87363284
 431886.15400629 434417.06933787 435055.1992594  441057.64478513
 448812.38071592 453733.52399413 456037.06457212 457713.45456995
 458827.35789148 461660.94618302 455001.19004925 454611.68312246
 452166.29802917 450662.31916803 450575.76639073 449677.75237534
 449006.87935519 444916.87623915 437721.74957046 435990.47467853
 435622.60817724 434421.62775666 424142.67946041 421513.43006745
 421264.60632119 420799.34395841 419371.1481722  409524.42278977
 406970.58226794 397848.36610792 389234.94350839 378034.69079697
 372093.35215962 356557.6

## Convert the label coordinates to pixel coordinates

These pixel coordinates are in reference to the tif file, not the png yet (if it has padding).

In [141]:
# Convert to pixel coordinates
pixel_coords = np.array([~raster_transform * (x, y) for x, y in zip(x_proj, y_proj)])
print(pixel_coords)

[[-5247.1933867  25213.32388087]
 [-4655.94962132 25209.18886388]
 [-3812.31298434 25217.36263417]
 ...
 [15880.24554621 17762.28222023]
 [14388.58932722 14003.82121736]
 [-3876.97579166 20974.25957172]]


## Add the new columns to the dataframe

Here is where we add to the x and y coordinates to account for the padding in the png file.

In [142]:
# Add the new columns to the dataframe
waterhole_labelled_df['x_proj'] = x_proj
waterhole_labelled_df['y_proj'] = y_proj
waterhole_labelled_df['pixel_col'] = pixel_coords[:, 0] + width_diff
waterhole_labelled_df['pixel_row'] = pixel_coords[:, 1] + height_diff

print(waterhole_labelled_df.head())

    Timestamp  Latitude  Longitude  Class        Observer  ObserverSi  \
0  2024-05-07 -13.65579   134.3538      2  Andrew Hoskins  Back Right   
1  2024-05-07 -13.65572   134.3702      2  Andrew Hoskins  Back Right   
2  2024-05-07 -13.65600   134.3936      3  Andrew Hoskins  Back Right   
3  2024-05-07 -13.65634   134.4219      2  Andrew Hoskins  Back Right   
4  2024-05-07 -13.65627   134.5256      2  Andrew Hoskins  Back Right   

  Water Type Chopper Water Type Satellite transectNu permanence    ID  \
0                NaN                River        m08          P         
1                NaN      River/Billabong        m08          P         
2                NaN            Billabong        m08     I or E         
3                NaN               Stream        m08     I or E  W269   
4                NaN            Billabong        m08     I or E         

  Wet Dry Chopper Wet Dry Satellite         x_proj        y_proj    pixel_col  \
0             NaN               Wet  4301

## Define the function to create the json file

Essentially we are just taking the pixel coordinates and the labels and creating a json file with the correct format.

In [143]:
def csv_to_labelme(x, y, 
                   labels=None, 
                   image_path=None, 
                   image_height=None, 
                   image_width=None):
    """
    Convert coordinate columns from a DataFrame to LabelMe JSON format.
    
    Args:
        x (pd.Series): Series containing x/longitude coordinates
        y (pd.Series): Series containing y/latitude coordinates
        labels (pd.Series, optional): Series containing point labels. Defaults to None
        image_path (str, optional): Path to the corresponding image file. Defaults to None
        image_height (int, optional): Height of the image in pixels. Defaults to None
        image_width (int, optional): Width of the image in pixels. Defaults to None
        
    Returns:
        dict: LabelMe formatted JSON
    """
    # Validate inputs
    if len(x) != len(y):
        raise ValueError("x and y coordinates must have the same length")
    if labels is not None and len(labels) != len(x):
        raise ValueError("labels must have the same length as coordinates")
    
    # Initialize LabelMe JSON structure
    labelme_json = {
        "version": "5.0.1",
        "flags": {},
        "shapes": [],
        "imagePath": os.path.basename(image_path) if image_path else "",
        "imageData": None,  # LabelMe stores base64 image data here, but we'll leave it empty
        "imageHeight": image_height,
        "imageWidth": image_width
    }
    
    # Convert each point to LabelMe shape
    point_size = 5  # Size of the point representation in pixels
    
    for i in range(len(x)):
        # Skip if coordinates are NaN
        if pd.isna(x[i]) or pd.isna(y[i]):
            continue
            
        shape = {
            "label": str(labels.iloc[i]) if labels is not None else "point",
            "points": [
                [float(x[i]) - point_size, float(y[i]) - point_size],  # Top-left
                [float(x[i]) + point_size, float(y[i]) + point_size]   # Bottom-right
            ],
            "group_id": None,
            "shape_type": "rectangle",
            "flags": {}
        }
        
        labelme_json['shapes'].append(shape)
    
    # Add creation time
    labelme_json['timeStamp'] = datetime.now().isoformat()
    
    return labelme_json

## Function to save the json file

In [144]:
# Convert the DataFrame to LabelMe JSON
def save_labelme_json(labelme_json, output_path):
    """Save the LabelMe JSON to file."""
    with open(output_path, 'w') as f:
        json.dump(labelme_json, f, indent=2)

## Run the csv to json function

The image path should be the name of the png file, and the output path of the save_labelme_json function should lead to where the png is saved.

In [145]:
# Convert and save
labelme_json = csv_to_labelme(
    x=waterhole_labelled_df['pixel_col'],
    y=waterhole_labelled_df['pixel_row'],
    labels=waterhole_labelled_df['Class'],
    image_path=tif_name + '.png',
    image_height=imported_raster.shape[0],
    image_width=imported_raster.shape[1]
)
    
# Save the LabelMe JSON file
save_labelme_json(labelme_json, f'{base_dir}/{tif_name}.json')
# print(labelme_json)